# 04 LSTM

Sequence model for Remaining Useful Life forecasting.

In [ ]:
from pathlib import Path
import sys
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.train import nasa_score

PROCESSED_DIR = ROOT / 'data' / 'processed'
MODELS_DIR = ROOT / 'models'
REPORTS_DIR = ROOT / 'reports' / 'figures'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

## 1) Load Sequence Data
Load preprocessed LSTM sequences from feature engineering output.

In [ ]:
seq_path = PROCESSED_DIR / 'fd001_sequences.npz'
data = np.load(seq_path, allow_pickle=True)
X_train = torch.from_numpy(data['X_train']).float()
y_train = torch.from_numpy(data['y_train']).float()
X_test = torch.from_numpy(data['X_test']).float()
y_test = torch.from_numpy(data['y_test']).float()

n_features = X_train.shape[-1]
seq_len = X_train.shape[1]

print(f'X_train: {X_train.shape}, y_train: {y_train.shape}')
print(f'X_test: {X_test.shape}, y_test: {y_test.shape}')
print(f'Sequence length: {seq_len}, Features: {n_features}')

## 2) Define LSTM Model
Create a recurrent architecture with LSTM and dense output layers.

In [ ]:
class LSTMPredictor(nn.Module):
    def __init__(self, n_features, hidden_size=64, num_layers=2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.2,
        )
        self.fc1 = nn.Linear(hidden_size, 32)
        self.fc2 = nn.Linear(32, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        last_hidden = lstm_out[:, -1, :]
        fc1_out = self.relu(self.fc1(last_hidden))
        pred = self.fc2(fc1_out)
        return pred.squeeze()

model = LSTMPredictor(n_features=n_features, hidden_size=64, num_layers=2).to(DEVICE)
print(f'Model:\n{model}')

## 3) Train LSTM Model
Fit the model for multiple epochs and track validation performance.

In [ ]:
train_ds = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', factor=0.5, patience=5)

num_epochs = 50
train_losses, val_losses = [], []

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0
    for x_batch, y_batch in train_loader:
        x_batch, y_batch = x_batch.to(DEVICE), y_batch.to(DEVICE)
        optimizer.zero_grad()
        pred = model(x_batch)
        loss = criterion(pred, y_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    epoch_loss /= len(train_loader)
    train_losses.append(epoch_loss)

    model.eval()
    with torch.no_grad():
        val_pred = model(X_test.to(DEVICE)).cpu().numpy()
        val_loss = float(np.sqrt(np.mean((val_pred - y_test.numpy())**2)))
        val_losses.append(val_loss)

    scheduler.step(val_loss)

    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1}/{num_epochs}: Train Loss={epoch_loss:.4f}, Val RMSE={val_loss:.4f}')

## 4) Evaluate and Save
Report test metrics and save trained model.

In [ ]:
model.eval()
with torch.no_grad():
    y_pred = model(X_test.to(DEVICE)).cpu().numpy()

rmse = float(np.sqrt(np.mean((y_pred - y_test.numpy())**2)))
mae = float(np.mean(np.abs(y_pred - y_test.numpy())))
score = float(nasa_score(y_test.numpy(), y_pred))

metrics = {'rmse': rmse, 'mae': mae, 'nasa_score': score, 'num_params': sum(p.numel() for p in model.parameters())}
print(f'Test RMSE: {rmse:.4f}')
print(f'Test MAE: {mae:.4f}')
print(f'NASA Score: {score:.2f}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].plot(train_losses, label='Train Loss')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].set_title('Training Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(val_losses, label='Val RMSE', color='orange')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('RMSE')
axes[0, 1].set_title('Validation RMSE')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].scatter(y_test.numpy(), y_pred, alpha=0.3, s=10)
axes[1, 0].plot([0, 130], [0, 130], linestyle='--', color='black', linewidth=1)
axes[1, 0].set_xlabel('True RUL')
axes[1, 0].set_ylabel('Predicted RUL')
axes[1, 0].set_title('Predicted vs True RUL')

residual = y_pred - y_test.numpy()
axes[1, 1].hist(residual, bins=40, color='#D95F02', edgecolor='black', alpha=0.7)
axes[1, 1].set_xlabel('Prediction Error')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('Residual Distribution')

plt.tight_layout()
plot_path = REPORTS_DIR / '07_lstm_diagnostics.png'
plt.savefig(plot_path, bbox_inches='tight')
plt.show()

torch.save(model.state_dict(), MODELS_DIR / 'lstm_model.pth')
(MODELS_DIR / 'lstm_metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')

print(f'\nSaved: {MODELS_DIR / "lstm_model.pth"}')
print(f'Saved: {MODELS_DIR / "lstm_metrics.json"}')
print(f'Saved: {plot_path}')